This script subclasses residential LoD2 buildings into single-family/semi-detached (SFH-DB), semi-detached buildings (SBD), terraced buildings (TB), and multi-family apartment buildings (MFH-AB) based on building size, height, and connectivity. Buildings are processed district by district, exported individually, and finally merged into a single GeoPackage.

In [ ]:
"""
LoD2 Residential Subclassification Script v6
============================================
Processes buildings sequentially, district by district.
Each district is exported as a separate GeoPackage.

  units_in_group == 1  → SFH-DB
  units_in_group == 2  → SBD
  units_in_group >  2  → TB

Priority: MFH-AB > TB > SBD > SFH-DB
"""

import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.ops import unary_union
from shapely.strtree import STRtree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
INPUT_PATH   = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_classified.gpkg"
VG250_PATH   = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
VG250_LAYER  = "vg250_krs"
OUTPUT_DIR   = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\Landkreise")

BUFFER_M                 = 0.05
HEIGHT_COL               = "measured_height"

THRESH_MFH_HEIGHT   = 9.4    # SFH-DB 90th percentile
THRESH_MFH_AREA_MIN = 129.6  # MFH-AB 10th percentile
THRESH_AREA_UPPER   = 190.8  # SFH-DB 90th percentile; buildings above this threshold are always classified as MFH-AB. Buildings below this threshold are classified as SFH-DB only if they are low-rise or height is unavailable.
THRESH_SBD_REL_AREA = 25.0   # Droin et al. (2023): at least 25% of the dissolved building area must belong to the building to be classified as SBD.

# ─────────────────────────────────────────────
# Residential subclassification
# ─────────────────────────────────────────────
def classify_residential(row):
    area     = row["area_m2"]
    units    = row["units_in_group"]
    rel_area = row["Rel_Area"]
    try:
        height = float(row[HEIGHT_COL])
    except (TypeError, ValueError, KeyError):
        height = np.nan

    # 1. MFH-AB (highest priority, regardless of neighboring buildings):
    #    a) large footprint (> 190.8 m²) → always MFH-AB
    #    b) footprint ≥ 129.6 m² and tall (≥ 9.4 m) → MFH-AB
    if area > THRESH_AREA_UPPER:
        return "MFH-AB"
    if (not np.isnan(height)
            and height >= THRESH_MFH_HEIGHT
            and area >= THRESH_MFH_AREA_MIN):
        return "MFH-AB"

    # 2. SFH-DB: standalone buildings only:
    #    - footprint < 129.6 m² → always SFH-DB
    #      (even if tall, as MFH-AB has already been assigned above)
    #    - footprint between 129.6 and 190.8 m²
    #      → only if low-rise (< 9.4 m) or height is unavailable
    if units == 1:
        if area < THRESH_MFH_AREA_MIN:
            return "SFH-DB"
        if area <= THRESH_AREA_UPPER and (np.isnan(height) or height < THRESH_MFH_HEIGHT):
            return "SFH-DB"

    # 3. SBD: exactly two buildings in the connected group
    #    and the relative area criterion is fulfilled
    if units == 2 and not np.isnan(rel_area) and rel_area >= THRESH_SBD_REL_AREA:
        return "SBD"

    # 4. TB: more than two buildings in the connected group
    if units > 2:
        return "TB"

    return "unclassified_res"


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load data
print(" Load data...")
gdf = gpd.read_file(INPUT_PATH)
if gdf.crs.is_geographic:
    gdf = gdf.to_crs(epsg=25832)
gdf = gdf.reset_index(drop=True)

landkreise = gpd.read_file(VG250_PATH, layer=VG250_LAYER).to_crs(gdf.crs)
landkreise = landkreise[landkreise["AGS"].astype(str).str.startswith("09")].copy().reset_index(drop=True)
print(f"   {len(landkreise)} Landkreise (Bayern)")

# 2. Only residential
res_mask = gdf["building_class"] == "residential"
gdf_res  = gdf[res_mask].copy().reset_index(drop=False).rename(columns={"index": "orig_idx"})
if "area_m2" not in gdf_res.columns:
    gdf_res["area_m2"] = gdf_res.geometry.area
print(f"   Residential: {len(gdf_res):,} | Andere: {(~res_mask).sum():,}")

print("\n🗺️  Centroid-Join...")
centroids = gpd.GeoDataFrame(
    {"orig_idx": gdf_res["orig_idx"]},
    geometry=gdf_res.geometry.centroid,
    crs=gdf_res.crs
)
joined = gpd.sjoin(
    centroids,
    landkreise[["AGS", "GEN", "geometry"]],
    how="left",
    predicate="within"
).rename(columns={"AGS": "lk_id", "GEN": "lk_name"})

missing = joined["lk_id"].isna()
if missing.sum() > 0:
    print(f"   {missing.sum()} Centroids außerhalb → nearest LK...")
    nearest = gpd.sjoin_nearest(
        centroids.loc[joined[missing].index],
        landkreise[["AGS", "GEN", "geometry"]],
        how="left"
).rename(columns={"AGS": "lk_id", "GEN": "lk_name"})
    nearest = nearest[~nearest.index.duplicated(keep="first")]
    joined.loc[missing, "lk_id"]   = nearest["lk_id"]
    joined.loc[missing, "lk_name"] = nearest["lk_name"]

gdf_res["lk_id"]   = joined["lk_id"].astype(str).values
gdf_res["lk_name"] = joined["lk_name"].values

lk_list = gdf_res[["lk_id", "lk_name"]].drop_duplicates().sort_values("lk_id")
print(f"   {len(lk_list)} Landkreise mit Gebäuden\n")

# 4. Process one district at a time
cols_needed = ["orig_idx", "geometry", "area_m2"] + \
              ([HEIGHT_COL] if HEIGHT_COL in gdf_res.columns else [])

for i, (_, lk_row) in enumerate(lk_list.iterrows()):
    lk_id   = lk_row["lk_id"]
    lk_name = lk_row["lk_name"]

    gdf_lk = gdf_res.loc[gdf_res["lk_id"] == lk_id, cols_needed].copy().reset_index(drop=True)
    n = len(gdf_lk)
    print(f"[{i+1:02d}/{len(lk_list)}] {lk_name} ({lk_id}) — {n:,} Gebäude", end=" ... ")

    # Connected Components
    buffered = gdf_lk.geometry.buffer(BUFFER_M).values
    tree = STRtree(buffered)
    rows, cols = [], []
    for idx, geom in enumerate(buffered):
        for j in tree.query(geom):
            if j <= idx:
                continue
            if buffered[j].intersects(geom):
                rows.append(idx)
                cols.append(j)

    if rows:
        data = np.ones(len(rows), dtype=np.int8)
        adj  = csr_matrix(
            (np.concatenate([data, data]),
             (np.concatenate([rows, cols]), np.concatenate([cols, rows]))),
            shape=(n, n)
        )
        _, labels = connected_components(adj, directed=False)
    else:
        labels = np.arange(n)

    gdf_lk["dissolved_id"]   = lk_id + "_" + labels.astype(str)
    gdf_lk["units_in_group"] = gdf_lk.groupby("dissolved_id")["dissolved_id"].transform("count")
    gdf_lk["Cons_Neigh"]     = gdf_lk["units_in_group"] - 1

    # Area_Dis / Perim_Dis
    group_geoms = {
        did: (grp.geometry.iloc[0] if len(grp) == 1 else unary_union(grp.geometry.values))
        for did, grp in gdf_lk.groupby("dissolved_id")
    }
    gdf_lk["Area_Dis"]  = gdf_lk["dissolved_id"].map({d: g.area   for d, g in group_geoms.items()})
    gdf_lk["Perim_Dis"] = gdf_lk["dissolved_id"].map({d: g.length for d, g in group_geoms.items()})
    gdf_lk["Rel_Area"]  = (gdf_lk["area_m2"] / gdf_lk["Area_Dis"] * 100).round(2)

    # Subclassification
    gdf_lk["res_subclass"] = gdf_lk.apply(classify_residential, axis=1)

    # Export
    safe_name = lk_name.replace(" ", "_").replace("/", "-")
    out_path  = OUTPUT_DIR / f"{lk_id}_{safe_name}.gpkg"

    if out_path.exists():
        print(f"⏭  bereits vorhanden, übersprungen")
        continue

    gdf_lk.to_file(out_path, driver="GPKG")

    # Stats
    counts = gdf_lk["res_subclass"].value_counts()
    stats  = "  ".join(f"{k}:{v}" for k, v in counts.items())
    print(f"✓  {stats}")

print(f"\n District done → {OUTPUT_DIR}")

# ─────────────────────────────────────────────
# 5. Merge all district GeoPackages
# ─────────────────────────────────────────────
FINAL_GPKG = OUTPUT_DIR.parent / "LoD2_2025_residential_types.gpkg"

print(f"\n Merge all districts → {FINAL_GPKG.name} ...")
gpkg_files = sorted(OUTPUT_DIR.glob("*.gpkg"))
print(f"   {len(gpkg_files)} files found")

parts = []
for f in gpkg_files:
    parts.append(gpd.read_file(f))

combined = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=parts[0].crs)
combined.to_file(FINAL_GPKG, driver="GPKG")
print(f"Done! {len(combined):,} buildings → {FINAL_GPKG}")